# plant-read-placement: GPU steps on Kaggle

Runs the three PlantCaduceus steps (embed, adapt the top layers, re-embed) and trains the two
contrastive heads, then packs the outputs for download. Everything else runs on your laptop.

**Before running**
1. *Settings > Accelerator:* **GPU T4 x2** (not P100).
2. *Settings > Internet:* **On** (Kaggle asks for a verified phone number).
3. *Add Input:* your private dataset containing `prp_windows.zip` (made with `scripts/pack_for_kaggle.sh`).

**Order:** run cells 1-5 first (about 15 minutes). If `gpu-check` prints OK, run the rest, or use
*Save Version > Save & Run All* so it continues in the background. Download `gpu_outputs.zip` from the
Output panel at the end.

In [ ]:
# 1. Which GPU did we get?
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# 2. Code at the latest commit (step out of the folder first, so re-running this cell can delete it)
%cd /kaggle/working
!rm -rf /kaggle/working/prp && git clone -q https://github.com/gbadedata/plant-read-placement /kaggle/working/prp
%cd /kaggle/working/prp
!git log --oneline -1

In [ ]:
# 3. Isolated Python 3.11 environment with PlantCaduceus's pinned versions (10-20 min)
import os, subprocess, sys

def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}  # keep Kaggle's sitecustomize out
    p = subprocess.Popen(cmd, shell=True, cwd="/kaggle/working/prp", stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed (exit {p.returncode}): {cmd}")

sh("bash scripts/kaggle_setup.sh")
PRP = "/tmp/prp-venv/bin/prp"

In [ ]:
# 4. Put the uploaded windows where the pipeline expects them
import glob, os, pathlib, zipfile
work = pathlib.Path("/kaggle/working/prp/work"); work.mkdir(exist_ok=True)
hits = glob.glob("/kaggle/input/**/train.npz", recursive=True)
if not hits:  # Kaggle kept the zip as a zip: extract it
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "No windows found: add your prp_windows dataset as notebook input"
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/working/windows_data")
    hits = glob.glob("/kaggle/working/windows_data/**/train.npz", recursive=True)
src = os.path.dirname(hits[0])
link = work / "windows"
if link.is_symlink():
    link.unlink()
if not link.exists():
    os.symlink(src, link)
print(src, "->", link, sorted(os.listdir(link)))

In [ ]:
# 5. Pre-flight: model loads, strand-invariant, fp16 agrees with fp32, layer names match, time estimate.
#    Stops here if anything is wrong.
sh(f"{PRP} gpu-check --encoder plantcad")

## Full run
Only continue if cell 5 printed `[gpu-check] OK`. Its time estimate is for one pass over all windows;
the two embedding passes each take about that long, and fine-tuning adds a similar amount.

In [ ]:
# 6. Frozen PlantCaduceus embeddings for every read (train, val, test)
sh(f"{PRP} embed --encoder plantcad")

In [ ]:
# 7. Adapt the top PlantCaduceus layers on noisy read windows (contrastive loss)
sh(f"{PRP} finetune --name plantcad_adapted")

In [ ]:
# 8. Re-embed every read with the adapted model
sh(f"{PRP} embed --encoder plantcad --checkpoint work/models/plantcad_adapted/encoder.pt --name plantcad_ft")

In [ ]:
# 9. Contrastive heads on both feature sets (GPU)
sh(f"{PRP} train-head --features plantcad")
sh(f"{PRP} train-head --features plantcad_ft")

In [ ]:
# 10. Pack everything the laptop needs; download gpu_outputs.zip from the Output panel
sh("cd work && /tmp/prp-venv/bin/python -m zipfile -c /kaggle/working/gpu_outputs.zip features models gpu_check.json")
sh("ls -lh /kaggle/working/gpu_outputs.zip")